In [1]:
import pandas as pd
import json
import numpy as np
import polars as pl
import requests
import plotly.express as px
import time
import datetime
import sys
import os
import matplotlib
import http.client
import urllib.parse
import pickle
import httpx
import csv

from tqdm.notebook import tqdm_notebook as tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pprint import pprint
from io import StringIO

from requests.adapters import ConnectionError, ReadTimeout, ReadTimeoutError
from urllib3.connection import NewConnectionError
from urllib3.util.retry import MaxRetryError

# Settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 100
pd.options.display.width = 20000
pd.set_option('display.float_format', lambda x: '%.4f' % x)
np.set_printoptions(suppress=True)
pl.Config.set_tbl_rows(100)

# from IPython.display import display, HTML
# display(HTML("<style>.container { width:90% !important; }</style>"))

polars.config.Config

In [2]:
def make_request(time_data: str, asset: str, request: str, params: dict):
    """

    :param time_data: at_time | hist | bulk_hist | list/...
    :param asset: stock | option | index
    :return: dict or raw csv
    """
    
    headers = {'Accept': "application/json"}
    url = f"http://127.0.0.1:25511/v2/{time_data}/{asset}/{request}".rstrip('/')

    success = False
    while not success:
        try:
            data = requests.get(url, headers=headers, params=params, timeout=60)
            if data.status_code == 474:
                print(f"{t} 474 Connection lost to Theta Data. It's going to sleep to 5 sec")
                time.sleep(5)
            elif ('Next-page' not in data.headers) and (data.status_code != 472): 
                print(f"{t} 'Next-page' not in the data.headers. It's going to sleep to 5 sec")
                print(data)
                print(data.content)
                time.sleep(5)
            else:
                success = True
        except (ConnectionError, NewConnectionError, MaxRetryError, ReadTimeout, ReadTimeoutError) as e:
            print(f"--------{params['root']}--------")
            print(e)
            time.sleep(5)

    if data.status_code == 471: 
        raise Exception(data.text)
    elif data.status_code == 472:
        return None, data.status_code
    elif data.status_code == 475:
        print(f"The server incorrect compile: {data.url}")
        return None, data.status_code

    if data.status_code != 200:
        print(params['root'], data.status_code, data.text)

    if ('use_csv' in list(params.keys())) and (params['use_csv'] == 'True'):
        if data.headers['Next-page'] != 'null':
            raise Exception(f"{params['root']} has an additional pages.")
        return data.content.decode('utf-8'), data.status_code
    else:
        return data.json(), data.status_code

# Options EOD

In [ ]:
# Ticker	Name	Days*
# GLD	SPDR GOLD TR GOLD SHS	MON,WED,FRI
# IWM	ISHARES TR RUSSELL 2000 ETF	MON,TUE,WED,THU,FRI
# QQQ	INVESCO QQQ TR UNIT SER 1	MON,TUE,WED,THU,FRI
# SLV	ISHARES SILVER TR ISHARES	MON,WED,FRI - слишком дешёвый
# SPY	SPDR S&P 500 ETF TR TR UNIT	MON,TUE,WED,T HU,FRI
# TLT	ISHARES TR 20 YR TR BD ETF	MON,WED,THU,FRI - слишком дешёвый
# UNG	UNITED STS NAT GAS FD LP UNIT PAR	WED,FRI - слишком дешёвый
# USO	UNITED STS OIL FD LP UNITS	WED,FRI - слишком дешёвый

In [3]:
def get_option_eod(t: str, start_date: str, end_date: str):
    if end_date < '20160101':
        return None
        
    params = {
        'root': t, 'exp': 0,
        'start_date': start_date, 'end_date': end_date,  
        'use_csv': 'True'
    }
    ans, status_code = make_request('bulk_hist', 'option', 'eod', params)
    
    if status_code == 472:
        print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
        return None
    elif status_code == 572:
        print(f"\033[1;91m {t} an error \033[0m")
        return None
    
    ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))
    
    return ans_df

In [4]:
load_year: str = '2024'

liquid_tickers = []
for core, folder, files in os.walk('data\stocks\daily'):
    for file in tqdm(files):
        cur_df = pl.read_parquet(os.path.join(core, file))
        if (cur_df['rol_50_vol'] > 1_000_000).any():
            liquid_tickers.append(file.split('_')[2])

already_tickers = [file.split('_')[2] for file in os.listdir('data/options') if load_year in file]
tickers_for_load = set(liquid_tickers) - set(already_tickers)
len(tickers_for_load)

  0%|          | 0/7804 [00:00<?, ?it/s]

1444

In [5]:
with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)
    
for t in tqdm(tickers_for_load, total=len(tickers_for_load)):
    dates = tickers_all_exp[t]['trade_dates']
    dates = pd.Series(dates, index=dates)[f'{load_year}0101':f'{load_year}1231'].values
    if len(dates) == 0:
        continue

    expirations = tickers_all_exp[t]['expirations']
    interseс = set(pd.Series(dates).astype(str)) & set(pd.Series(expirations).astype(str))
    if len(interseс) == 0:
        continue

    final_df = pl.DataFrame()   
    with ThreadPoolExecutor(max_workers=24) as executor:
        futures = {
            executor.submit(get_option_eod, t, start, end): 
            (t, start, end) for start, end in zip(dates, dates)
        }
        for future in tqdm(as_completed(futures), total=len(dates)):
            try:
                cur_df = future.result()
                if (cur_df is None) or len(cur_df) == 0: 
                    continue
                final_df = pl.concat([final_df, cur_df])
            except Exception as e:
                print(cur_df)
                print(e)
                raise Exception
    try:
        if (final_df is None) or len(final_df) == 0: 
            continue
        final_df = final_df.sort('date', 'expiration', 'right', 'strike')\
            .with_columns(pl.col('date').cast(pl.String).str.strptime(pl.Date, '%Y%m%d', strict=False))
        for year in final_df['date'].dt.year().unique():
            cur_df = final_df.filter(pl.col('date').dt.year() == year)\
                .unique(subset=['expiration', 'date', 'right', 'strike'])
            min_date = cur_df['date'].min().strftime('%Y%m%d')
            max_date = cur_df['date'].max().strftime('%Y%m%d')

            cur_df.write_parquet(f'data/options/{min_date}_{max_date}_{t}_opts.parquet')
    except Exception as e:
        print(final_df)
        print(cur_df)
        print(e)
        raise Exception

  0%|          | 0/1444 [00:00<?, ?it/s]

  0%|          | 0/203 [00:00<?, ?it/s]

 INVO 20240102 - 20240102 don't have data 
 INVO 20240110 - 20240110 don't have data 
 INVO 20240103 - 20240103 don't have data 
 INVO 20240108 - 20240108 don't have data 
 INVO 20240104 - 20240104 don't have data 
 INVO 20240109 - 20240109 don't have data 
 INVO 20240112 - 20240112 don't have data 
 INVO 20240105 - 20240105 don't have data 
 INVO 20240111 - 20240111 don't have data 
 INVO 20240116 - 20240116 don't have data 
 INVO 20240117 - 20240117 don't have data 
 INVO 20240118 - 20240118 don't have data 
 INVO 20240122 - 20240122 don't have data 
 INVO 20240119 - 20240119 don't have data 
 INVO 20240125 - 20240125 don't have data 
 INVO 20240123 - 20240123 don't have data 
 INVO 20240126 - 20240126 don't have data 
 INVO 20240129 - 20240129 don't have data 
 INVO 20240130 - 20240130 don't have data 
 INVO 20240131 - 20240131 don't have data 
 INVO 20240201 - 20240201 don't have data 
 INVO 20240202 - 20240202 don't have data 
 INVO 20240205 - 20240205 don't have data 
 INVO 20240

  0%|          | 0/224 [00:00<?, ?it/s]

  0%|          | 0/224 [00:00<?, ?it/s]

...

  0%|          | 0/224 [00:00<?, ?it/s]

 ISPO 20240103 - 20240103 don't have data 
 ISPO 20240102 - 20240102 don't have data 
 ISPO 20240105 - 20240105 don't have data 
 ISPO 20240104 - 20240104 don't have data 
 ISPO 20240108 - 20240108 don't have data 
 ISPO 20240110 - 20240110 don't have data 
 ISPO 20240112 - 20240112 don't have data 
 ISPO 20240116 - 20240116 don't have data 
 ISPO 20240118 - 20240118 don't have data 
 ISPO 20240119 - 20240119 don't have data 
 ISPO 20240124 - 20240124 don't have data 
 ISPO 20240129 - 20240129 don't have data 
 ISPO 20240130 - 20240130 don't have data 
 ISPO 20240131 - 20240131 don't have data 
 ISPO 20240201 - 20240201 don't have data 
 ISPO 20240123 - 20240123 don't have data 
 ISPO 20240122 - 20240122 don't have data 
 ISPO 20240125 - 20240125 don't have data 
 ISPO 20240109 - 20240109 don't have data 
 ISPO 20240117 - 20240117 don't have data 
 ISPO 20240111 - 20240111 don't have data 
 ISPO 20240126 - 20240126 don't have data 
 ISPO 20240202 - 20240202 don't have data 
 ISPO 20240

  0%|          | 0/224 [00:00<?, ?it/s]

 NBBK 20240102 - 20240102 don't have data 
 NBBK 20240105 - 20240105 don't have data 
 NBBK 20240104 - 20240104 don't have data 
 NBBK 20240108 - 20240108 don't have data 
 NBBK 20240103 - 20240103 don't have data 
 NBBK 20240109 - 20240109 don't have data 
 NBBK 20240110 - 20240110 don't have data 
 NBBK 20240111 - 20240111 don't have data 
 NBBK 20240112 - 20240112 don't have data 
 NBBK 20240117 - 20240117 don't have data 
 NBBK 20240116 - 20240116 don't have data 
 NBBK 20240118 - 20240118 don't have data 
 NBBK 20240122 - 20240122 don't have data 
 NBBK 20240119 - 20240119 don't have data 
 NBBK 20240124 - 20240124 don't have data 
 NBBK 20240129 - 20240129 don't have data 
 NBBK 20240126 - 20240126 don't have data 
 NBBK 20240125 - 20240125 don't have data 
 NBBK 20240123 - 20240123 don't have data 
 NBBK 20240130 - 20240130 don't have data 
 NBBK 20240202 - 20240202 don't have data 
 NBBK 20240205 - 20240205 don't have data 
 NBBK 20240131 - 20240131 don't have data 
 NBBK 20240

  0%|          | 0/224 [00:00<?, ?it/s]

# Stocks 1 Minute

In [7]:
def get_stock_1m(t: str, start_date: str, end_date: str):
    if end_date < '20160101':
        return None
        
    print(start_date, end_date)
    params = {
        'root': t, 'exp': 0,
        'start_date': start_date, 'end_date': end_date,  
        'use_csv': 'True', 'ivl': '60000'
    }
    ans, status_code = make_request('hist', 'stock', 'ohlc', params)
    
    if status_code == 472:
        print(f"\033[1;91m {t} {start_date} - {end_date} don't have data \033[0m")
        return None
    elif status_code == 572:
        print(f"\033[1;91m {t} an error \033[0m")
        return None
    
    ans_df = pl.read_csv(StringIO(ans)).with_columns(pl.lit(t).alias('ticker'))
    
    return ans_df

In [8]:
start_range = '20160101'
end_range = datetime.datetime.now().strftime('%Y%m%d')

weekly_start = list(pd.date_range(start_range, end_range, freq='W-MON').strftime('%Y%m%d')[::-1])
weekly_end = list(pd.date_range(start_range, end_range, freq='W-SUN').strftime('%Y%m%d')[::-1])
weekly_end.extend([end_range])
weekly_end = np.sort(weekly_end)[::-1]

with open('data/all_tickers_expir.pickle', 'rb') as f:
    tickers_all_exp = pickle.load(f)

# ['SPY', 'GLD', 'IWM', 'QQQ', 'SLV', 'TLT', 'UNG', 'USO']
need_tickers = ['SPY', 'GLD', 'IWM', 'QQQ', 'SLV', 'TLT', 'UNG', 'USO']
for t in tqdm(need_tickers, total=len(need_tickers)):
    final_df = pl.DataFrame()    
    start_time = time.time()

    count = 0
    with ThreadPoolExecutor(max_workers=12) as executor:
        futures = {
            executor.submit(get_stock_1m, t, weekly_start, weekly_end): 
            (t, weekly_start, weekly_end) for weekly_start, weekly_end in zip(weekly_start, weekly_end)
        }
        for future in tqdm(as_completed(futures), total=len(weekly_end)):
            cur_df = future.result()
            if cur_df is None: 
                continue
            final_df = pl.concat([final_df, cur_df])

            if len(final_df) == 0:
                count = 0
                continue

            count += 1
            if count > 10:
                final_df = final_df.sort('date')
                final_df.write_parquet(f'data/stocks/{start_range}_{end_range}_{t}_opts.parquet')
                count = 0

    final_df = final_df.sort('date')
    final_df.write_parquet(f'data/stocks/{start_range}_{end_range}_{t}_opts.parquet')

  0%|          | 0/8 [00:00<?, ?it/s]

2024111820241111 20241117
 20241121
20241104 20241110
20241028 20241103
20241021 20241027
20241014 20241020
20241007 20241013
20240930 20241006
20240923 20240929
20240916 20240922
20240909 20240915
20240902 20240908


  0%|          | 0/465 [00:00<?, ?it/s]

20240826 20240901
20240819 20240825
20240812 20240818
20240805 20240811
20240729 20240804
20240722 20240728
20240715 20240721
20240708 20240714
20240701 20240707
20240624 20240630
20240617 20240623
20240610 20240616
20240603 20240609
20240527 20240602
20240520 20240526
20240513 20240519
20240506 20240512
20240429 20240505
20240422 20240428
20240415 20240421
20240408 20240414
20240401 20240407
20240325 20240331
20240318 20240324
20240311 20240317
20240304 20240310
20240226 20240303
20240219 20240225
20240212 20240218
20240205 20240211
20240129 20240204
20240122 20240128
20240115 20240121
20240108 20240114
20240101 20240107
20231225 20231231
20231218 20231224
20231211 20231217
20231204 20231210
20231127 20231203
20231120 20231126
20231113 20231119
20231106 20231112
20231030 20231105
20231023 20231029
20231016 20231022
20231009 20231015
20231002 20231008
20230925 20231001
20230918 20230924
20230911 20230917
20230904 20230910
20230828 20230903
20230821 20230827
20230814 20230820
20230807 2

  0%|          | 0/465 [00:00<?, ?it/s]

...

20240826 20240901
20240819 20240825
20240812 20240818
20240805 20240811
20240729 20240804
20240722 20240728
20240715 20240721
20240708 20240714
20240701 20240707
20240624 20240630
20240617 20240623
20240610 20240616
20240603 20240609
20240527 20240602
20240520 20240526
20240513 20240519
20240506 20240512
20240429 20240505
20240422 20240428
20240415 20240421
20240408 20240414
20240401 20240407
20240325 20240331
20240318 20240324
20240311 20240317
20240304 20240310
20240226 20240303
20240219 20240225
20240212 20240218
20240205 20240211
20240129 20240204
20240122 20240128
20240115 20240121
20240108 20240114
20240101 20240107
20231225 20231231
20231218 20231224
20231211 20231217
20231204 20231210
20231127 20231203
20231120 20231126
20231113 20231119
20231106 20231112
20231030 20231105
20231023 20231029
20231016 20231022
20231009 20231015
20231002 20231008
20230925 20231001
20230918 20230924
20230911 20230917
20230904 20230910
20230828 20230903
20230821 20230827
20230814 20230820
20230807 2

  0%|          | 0/465 [00:00<?, ?it/s]

20240826 20240901
20240819 20240825
20240812 20240818
20240805 20240811
20240729 20240804
20240722 20240728
20240715 20240721
20240708 20240714
20240701 20240707
20240624 20240630
20240617 20240623
20240610 20240616
20240603 20240609
20240527 20240602
20240520 20240526
20240513 20240519
20240506 20240512
20240429 20240505
20240422 20240428
20240415 20240421
20240408 20240414
20240401 20240407
20240325 20240331
20240318 20240324
20240311 20240317
20240304 20240310
20240226 20240303
20240219 20240225
20240212 20240218
20240205 20240211
20240129 20240204
20240122 20240128
20240115 20240121
20240108 20240114
20240101 20240107
20231225 20231231
20231218 20231224
20231211 20231217
20231204 20231210
20231127 20231203
20231120 20231126
20231113 20231119
20231106 20231112
20231030 20231105
20231023 20231029
20231016 20231022
20231009 20231015
20231002 20231008
20230925 20231001
20230918 20230924
20230911 20230917
20230904 20230910
20230828 20230903
20230821 20230827
20230814 20230820
20230807 2

  0%|          | 0/465 [00:00<?, ?it/s]

20240826 20240901
20240819 20240825
20240812 20240818
20240805 20240811
20240729 20240804
20240722 20240728
20240715 20240721
20240708 20240714
20240701 20240707
20240624 20240630
20240617 20240623
20240610 20240616
20240603 20240609
20240527 20240602
20240520 20240526
20240513 20240519
20240506 20240512
20240429 20240505
20240422 20240428
20240415 20240421
20240408 20240414
20240401 20240407
20240325 20240331
20240318 20240324
20240311 20240317
20240304 20240310
20240226 20240303
20240219 20240225
20240212 20240218
20240205 20240211
20240129 20240204
20240122 20240128
20240115 20240121
20240108 20240114
20240101 20240107
20231225 20231231
20231218 20231224
20231211 20231217
20231204 20231210
20231127 20231203
20231120 20231126
20231113 20231119
20231106 20231112
20231030 20231105
20231023 20231029
20231016 20231022
20231009 20231015
20231002 20231008
20230925 20231001
20230918 20230924
20230911 20230917
20230904 20230910
20230828 20230903
20230821 20230827
20230814 20230820
20230807 2